In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

Data Preprosseing


In [2]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datasets/kc_house_data.csv')

In [3]:
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [4]:
df.shape

(21613, 21)

In [5]:
df = df.drop(['id', 'date'], axis=1)

In [6]:
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [7]:
df.shape

(21613, 19)

In [8]:
df.isnull().sum()

,0
price,0
bedrooms,0
bathrooms,0
sqft_living,0
sqft_lot,0
floors,0
waterfront,0
view,0
condition,0
grade,0


Define Target and Feature

In [9]:
x = df.drop('price', axis=1)
y = df['price']

Handling Catogorical data

In [10]:
x = pd.get_dummies(x,columns =['zipcode'],drop_first=True)

In [11]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

Find Besst Model


In [12]:
def model_evaluation(model):
    # Train the model
    model.fit(x_train, y_train)

    # Predictions
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # Metrics
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)

    r2 = r2_score(y_test, y_test_pred)

    # Print results
    print("Training MAE           :", train_mae)
    print("Test MAE               :", test_mae)
    print("Training RMSE          :", train_rmse)
    print("Test RMSE              :", test_rmse)
    print("R² Score (Test)        :", r2)

    # Cross-validation
    cv_scores = cross_val_score(model, x, y, cv=5, scoring='r2')
    print("Mean CV R² Score       :", cv_scores.mean())
    print("CV R² Scores           :", cv_scores)


Model Comparison


In [13]:
def compare_regression_models(models, x_train, x_test, y_train, y_test):
    results = []

    for name, model in models.items():
        # Train model
        model.fit(x_train, y_train)
        # Predict
        y_pred = model.predict(x_test)

        # Calculate metrics
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "Model": name,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2 Score": r2
        })

    return pd.DataFrame(results)


models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=500, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=500, random_state=42)
}

In [14]:
model = LinearRegression()
model_evaluation(model)

Training MAE           : 95089.95759504648
Test MAE               : 98749.44998057563
Training RMSE          : 158237.38738615502
Test RMSE              : 170912.00136730477
R² Score (Test)        : 0.8067763759688937
Mean CV R² Score       : 0.8043815879638284
CV R² Scores           : [0.79367054 0.78857915 0.80319307 0.82460165 0.81186353]


In [15]:
model = Ridge()
model_evaluation(model)

Training MAE           : 95226.09800490247
Test MAE               : 98855.8152253587
Training RMSE          : 158319.2822917645
Test RMSE              : 171062.75620753472
R² Score (Test)        : 0.8064353555222176
Mean CV R² Score       : 0.8042747298875005
CV R² Scores           : [0.79362818 0.78841828 0.80303455 0.8247716  0.81152104]


In [16]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model_evaluation(model)

Training MAE           : 25849.00015519729
Test MAE               : 72466.52817866506
Training RMSE          : 47411.17646024942
Test RMSE              : 149497.5701979583
R² Score (Test)        : 0.8521629213822263
Mean CV R² Score       : 0.8769136376678521
CV R² Scores           : [0.87632631 0.88433801 0.87926571 0.86597499 0.87866318]


In [17]:
model = GradientBoostingRegressor(n_estimators=100, random_state=42)
model_evaluation(model)

Training MAE           : 71419.96648484703
Test MAE               : 79695.8326742716
Training RMSE          : 110649.22596078874
Test RMSE              : 144344.89172662364
R² Score (Test)        : 0.862178192285527
Mean CV R² Score       : 0.8697511014445484
CV R² Scores           : [0.8807107  0.86816547 0.8665955  0.8640515  0.86923234]


In [18]:
model = XGBRegressor(n_estimators=100, random_state=42)
model_evaluation(model)

Training MAE           : 41632.48356302415
Test MAE               : 71418.17898811589
Training RMSE          : 58058.87969228809
Test RMSE              : 146368.8041946539
R² Score (Test)        : 0.8582861968517161
Mean CV R² Score       : 0.878821258231387
CV R² Scores           : [0.87194062 0.8804547  0.88974857 0.87390011 0.87806229]


In [19]:
result_df = compare_regression_models(models, x_train, x_test, y_train, y_test)
print(result_df)

               Model           MAE           MSE           RMSE  R2 Score
0  Linear Regression  98749.449981  2.921091e+10  170912.001367  0.806776
1   Ridge Regression  98855.815225  2.926247e+10  171062.756208  0.806435
2      Random Forest  72466.528179  2.234952e+10  149497.570198  0.852163
3  Gradient Boosting  69656.858927  1.746052e+10  132138.260468  0.884503
4            XGBoost  71777.208503  2.154657e+10  146787.502632  0.857474


In [20]:
model = XGBRegressor(n_estimators=100, random_state=42)
model.fit(x_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [27]:
# This is the input for a new house you want to predict
new_house = {
    "bedrooms": 3,
    "bathrooms": 1.75,
    "sqft_living": 3860,
    "sqft_lot": 9000,
    "floors": 1,
    "waterfront": 0,
    "view": 2,
    "condition": 3,
    "grade": 9,
    "sqft_above": 1930,
    "sqft_basement": 1930,
    "yr_built": 1970,
    "yr_renovated": 0,
    "zipcode": 98155,
    "lat": 47.7431,
    "long": -122.29,
    "sqft_living15": 2960,
    "sqft_lot15": 9000
}



# 1️⃣ Convert new house to DataFrame
new_house_df = pd.DataFrame([new_house])

# 2️⃣ One-hot encode 'zipcode' to match training data
new_house_df = pd.get_dummies(new_house_df, columns=['zipcode'], drop_first=True)

# 3️⃣ Add missing dummy columns (those present in training but missing in new house)
for col in x_train.columns:
    if col not in new_house_df.columns:
        new_house_df[col] = 0

# 4️⃣ Reorder columns to match training data
new_house_df = new_house_df[x_train.columns]

# 6️⃣ Predict price
predicted_price = model.predict(new_house_df)
print("Predicted Price: $", predicted_price[0])


Predicted Price: $ 788347.6
